<a href="https://colab.research.google.com/github/shannon112/Poneglyph/blob/main/Pytorch_MNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#@title Import Dependencies

import torch
import torch.nn as nn
import torchvision.datasets as dsets
import torchvision.transforms as transforms
from torch.autograd import Variable
import torch.nn.functional as F

In [3]:
#@title Define Hyperparameters

input_size = 784 # img_size = (28,28) ---> 28*28=784 in total
hidden_size = 500 # number of nodes at hidden layer
num_classes = 10 # number of output classes discrete range [0,9]
num_epochs = 5 # number of times which the entire dataset is passed throughout the model
batch_size = 100 # the size of input data took for one iteration
lr = 1e-3 # size of step

In [4]:
#@title Downloading MNIST data

train_data = dsets.MNIST(root = './data', train = True,
                        transform = transforms.ToTensor(), download = True)

test_data = dsets.MNIST(root = './data', train = False,
                       transform = transforms.ToTensor())

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9.91M/9.91M [00:00<00:00, 60.6MB/s]


Extracting ./data/MNIST/raw/train-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28.9k/28.9k [00:00<00:00, 1.70MB/s]


Extracting ./data/MNIST/raw/train-labels-idx1-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 1.65M/1.65M [00:00<00:00, 14.5MB/s]


Extracting ./data/MNIST/raw/t10k-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 4.54k/4.54k [00:00<00:00, 3.14MB/s]

Extracting ./data/MNIST/raw/t10k-labels-idx1-ubyte.gz to ./data/MNIST/raw



In [5]:
#@title Loading the data using dataloader

train_gen = torch.utils.data.DataLoader(dataset = train_data,
                                             batch_size = batch_size,
                                             shuffle = True)

test_gen = torch.utils.data.DataLoader(dataset = test_data,
                                      batch_size = batch_size,
                                      shuffle = False)

In [12]:
#@title Define model class

class NnNet(nn.Module): # nn.Module is a base class, NnNet inherits from nn.Module
  def __init__(self, input_size, hidden_size, num_classes):
    super().__init__() # calling methods from the parent class. initializes the parent class.
    self.fc1 = nn.Linear(input_size, hidden_size)
    self.relu = nn.ReLU()
    self.fc2 = nn.Linear(hidden_size, num_classes)

  def forward(self,x):
    out = self.fc1(x)
    out = self.relu(out)
    out = self.fc2(out)
    return out

# https://github.com/pytorch/examples/blob/main/mnist/main.py
class CnnNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1, 1) # in_channels, out_channels, kernel_size, stride=1, padding=0
        self.conv2 = nn.Conv2d(32, 64, 3, 1, 1)
        self.relu = nn.ReLU()
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(input_size*64//4, hidden_size)
        self.fc2 = nn.Linear(hidden_size, num_classes)
        self.max_pool2d = nn.MaxPool2d(2)

    def forward(self, x):
        # from (batch_size, 784)
        x = x.view(-1, 1, 28, 28)  # to (batch_size, 1, 28, 28)
        x = self.conv1(x) # to (batch_size, 32, 28, 28)
        x = self.relu(x)
        x = self.conv2(x) # to (batch_size, 64, 28, 28)
        x = self.relu(x)
        x = self.max_pool2d(x) # to (batch_size, 64, 14, 14)
        x = self.dropout1(x)
        x = torch.flatten(x, 1) # to (batch_size, 64x14x14)
        x = self.fc1(x) # to (batch_size, 100)
        x = self.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x) # to (batch_size, 10)
        output = F.log_softmax(x, dim=1)
        return output

In [13]:
#@title Build the model

net = CnnNet(input_size, hidden_size, num_classes)
if torch.cuda.is_available():
  net.cuda()

In [14]:
#@title Define loss-function & optimizer

loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam( net.parameters(), lr=lr)

In [16]:
#@title Training the model

for epoch in range(num_epochs):
  for i ,(images,labels) in enumerate(train_gen):
    images = Variable(images.view(-1,28*28)).cuda()
    labels = Variable(labels).cuda()

    optimizer.zero_grad()
    outputs = net(images)
    loss = loss_function(outputs, labels)
    loss.backward()
    optimizer.step()

    if (i+1) % 100 == 0:
      print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_data)//batch_size}], Loss: {loss.item()}')

Epoch [1/5], Step [100/600], Loss: 0.03887354955077171
Epoch [1/5], Step [200/600], Loss: 0.017345130443572998
Epoch [1/5], Step [300/600], Loss: 0.0014875681372359395
Epoch [1/5], Step [400/600], Loss: 0.026995040476322174
Epoch [1/5], Step [500/600], Loss: 0.016959652304649353
Epoch [1/5], Step [600/600], Loss: 0.07334556430578232
Epoch [2/5], Step [100/600], Loss: 0.012775352224707603
Epoch [2/5], Step [200/600], Loss: 0.002222534501925111
Epoch [2/5], Step [300/600], Loss: 0.01968720369040966
Epoch [2/5], Step [400/600], Loss: 0.0020355244632810354
Epoch [2/5], Step [500/600], Loss: 0.011135493405163288
Epoch [2/5], Step [600/600], Loss: 0.0006056554848328233
Epoch [3/5], Step [100/600], Loss: 0.003933109808713198
Epoch [3/5], Step [200/600], Loss: 0.001365845208056271
Epoch [3/5], Step [300/600], Loss: 0.0025321871507912874
Epoch [3/5], Step [400/600], Loss: 0.007989373058080673
Epoch [3/5], Step [500/600], Loss: 0.052602194249629974
Epoch [3/5], Step [600/600], Loss: 0.0552372708

In [46]:
#@title Evaluating the accuracy of the model

correct = 0
total = 0
for images,labels in test_gen:
  images = Variable(images.view(-1,28*28)).cuda()
  labels = labels.cuda()

  output = net(images)
  _, predicted = torch.max(output,1)
  correct += (predicted == labels).sum()
  total += labels.size(0)

print('Accuracy of the model: %.3f %%' %((100*correct)/(total+1)))

Accuracy of the model: 98.720 %
